# Avviare il Progetto su Google Colab (via GitHub)
Questo notebook scarica automaticamente il tuo codice più aggiornato da GitHub.
Ogni volta che fai `git push` dal tuo PC, ti basterà eseguire queste celle su Colab per avere la versione più recente e far partire il training sulla GPU.

### 1. Scarica / Aggiorna il codice da GitHub
Questa cella clona la repository se non esiste ancora nella macchina virtuale di Colab, oppure fa un `git pull` per scaricare le ultime modifiche se l'hai già clonata in precedenza.

In [ ]:
import os

repo_url = "https://github.com/giorgio-di-dio/vessel-project.git"
repo_name = "vessel-project"

if not os.path.exists(repo_name):
    print(f"Clonazione della repository...")
    # L'opzione -b clona direttamente il branch specifico
    !git clone -b main {repo_url}
else:
    print("Repository già presente.")

# Spostati nella cartella del progetto
%cd {repo_name}

# Se la cartella esisteva già, scarica le novità e spostati sul branch corretto
!git fetch origin

!git pull origin main

Clonazione della repository (branch: main)...
Cloning into 'vessel-project'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 121 (delta 59), reused 103 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 6.96 MiB | 23.31 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/vessel-project
Allineamento al branch: main...
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/giorgio-di-dio/vessel-project
 * branch            main       -> FETCH_HEAD
Already up to date.


### 2. Installa le librerie necessarie
Installa le dipendenze dal file `requirements.txt`.

In [2]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.9 MB/s eta 0:00:00


### 3. Salvare i risultati (Modelli allenati) su Google Drive
**ATTENZIONE IMPORTANTE:** Dato che stiamo usando GitHub per il codice, Colab cancellerà tutto (compresi i modelli allenati e i log) quando spegnerà la macchina virtuale!

Per non perdere ore di training, colleghiamo Google Drive SOLO per salvare gli output. Nel tuo `config.py`, assicurati che `OUTPUT_DIR` punti a una cartella di Drive se ti trovi su Colab, oppure al termine del training esegui la cella qui sotto per copiare a mano la cartella `output` sul tuo Drive.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 4. Avvia il Training!
Ora fai partire il tuo codice.

In [4]:
!python main.py

[Config] Verificata/Creata directory: /content/vessel-project/data/raw
[Config] Verificata/Creata directory: /content/vessel-project/data/processed
[Config] Verificata/Creata directory: /content/vessel-project/output/models
[Config] Verificata/Creata directory: /content/vessel-project/output/results
  VESSEL SEGMENTATION - TRAINING U-Net
  Device   : CUDA
  Epochs   : 50
  LR       : 0.0001
  Batch    : 8  |  Patch: 512x512
[Kaggle] Richiesta/Verifica dataset: nikitamanaenkov/fundus-image-dataset-for-vessel-segmentation
Using Colab cache for faster access to the 'fundus-image-dataset-for-vessel-segmentation' dataset.
[Dataset] Modalità='train' | Coppie valide=600 | patch_size=512x512 | Augmentation=True
[Dataset] Modalità='test' | Coppie valide=200 | patch_size=512x512 | Augmentation=False
[DataLoader] Train=540 | Val=60 | Test=200 immagini
[DataLoader] batch_size=8 (train) | 1 (val/test)
/content/vessel-project/main.py:94: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is depreca

### 5. Backup degli output (da eseguire A FINE training)
Esegui questa cella quando `main.py` ha finito, così ti salvi tutti i pesi `.pth` e le immagini sul tuo Drive.

In [5]:
import os
import glob

# Cerca il file CSV dei log per estrarre il timestamp della run
log_files = glob.glob("output/logs/run_*_epochs.csv")

if log_files:
    # Prendi l'ultimo file creato se ce n'è più di uno
    latest_log = max(log_files, key=os.path.getctime)
    # Estrae il timestamp dal nome del file (es. run_20260516_223915_epochs.csv -> 20260516_223915)
    run_timestamp = os.path.basename(latest_log).replace("run_", "").replace("_epochs.csv", "")
    print(f"Timestamp rilevato dalla run: {run_timestamp}")
else:
    # Fallback al tempo corrente se non trova log
    import datetime
    from zoneinfo import ZoneInfo
    run_timestamp = datetime.datetime.now(tz=ZoneInfo('Europe/Rome')).strftime("%Y%m%d_%H%M")
    print(f"Nessun log trovato. Uso timestamp attuale: {run_timestamp}")

# Costruisci il percorso di destinazione su Google Drive
destination_path = f"/content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/{run_timestamp}"

# Crea la cartella su Google Drive
print(f"Creazione della cartella di destinazione: {destination_path}")
!mkdir -p "{destination_path}"

# Copia i contenuti della cartella 'output'
print(f"Copia dei risultati nella cartella: {destination_path}")
!cp -r output/* "{destination_path}"

print("Salvataggio su Drive completato!")

Timestamp rilevato dalla run: 20260517_1240
Creazione della cartella di destinazione: /content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/20260517_1240
Copia dei risultati nella cartella: /content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/20260517_1240
Salvataggio su Drive completato!


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

# 1. Definisci la cartella radice su Drive
drive_output_base = "/content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/*"

# Cerca tutte le cartelle generate dentro output_pesi
all_folders = [f for f in glob.glob(drive_output_base) if os.path.isdir(f)]

if not all_folders:
    print("Errore: Nessuna cartella trovata dentro output_pesi su Google Drive.")
else:
    # Ordina alfabeticamente al contrario (la data più recente va all'indice [0])
    all_folders_sorted = sorted(all_folders, reverse=True)
    latest_folder = all_folders_sorted[0]

    folder_name = os.path.basename(latest_folder)
    print(f"-> Cartella più recente individuata: {folder_name}")

    # Estraiamo la data e l'ora dal nome della cartella (es. 20260517_0500)
    try:
        parti_nome = folder_name.split("_")
        data_raw = parti_nome[0]  # YYYYMMDD
        ora_raw = parti_nome[1]   # HHMM

        # Formattiamo per renderlo leggibile (es. 17/05/2026 - 05:00)
        data_formattata = f"{data_raw[6:8]}/{data_raw[4:6]}/{data_raw[0:4]}"
        ora_formattata = f"{ora_raw[0:2]}:{ora_raw[2:4]}"
        info_run = f"{data_formattata} alle {ora_formattata}"
    except Exception:
        info_run = folder_name # Fallback se il nome ha strutture strane

    # Costruisci il percorso dei log specifico per QUESTA cartella
    log_path_pattern = os.path.join(latest_folder, "logs", "run_*_epochs.csv")
    log_files = glob.glob(log_path_pattern)

    if not log_files:
        print(f"Errore: Nessun file CSV di log trovato in: {log_path_pattern}")
    else:
        # Ordina anche i log interni per sicurezza e prendi il primo
        latest_log = sorted(log_files, reverse=True)[0]
        print(f"Analisi del file di log: {os.path.basename(latest_log)}")

        # 2. Carica i dati
        df = pd.read_csv(latest_log)

        # Gestione colonna epoche
        if 'epoch' not in df.columns:
            df['epoch'] = df.index + 1

        # 3. Creazione dei Plot
        metrics = ['Loss', 'Dice', 'IoU']
        fig, axes = plt.subplots(1, 3, figsize=(20, 5))
        plt.suptitle(f'Trend delle Metriche - Run del {info_run}', fontsize=22, y=1.02)

        for i, metric in enumerate(metrics):
            train_col = f'train_{metric.lower()}'
            val_col = f'val_{metric.lower()}'

            if train_col in df.columns and val_col in df.columns:
                axes[i].plot(df['epoch'], df[train_col], label=f'Train {metric}', marker='o', markersize=3)
                axes[i].plot(df['epoch'], df[val_col], label=f'Val {metric}', marker='x', markersize=3)
                axes[i].set_title(f'Trend della {metric}',fontsize=13)
                axes[i].set_xlabel('Epoca')
                axes[i].set_ylabel(metric)
                axes[i].legend()
                axes[i].grid(True)

                # Scala dinamica per la Loss, fissa 0-1 per Dice e IoU
                if metric == 'Loss':
                    axes[i].set_ylim(0, max(df[train_col].max(), df[val_col].max()) * 1.1)
                else:
                    axes[i].set_ylim(0, 1.05)

                axes[i].set_xlim(0, 1.02 * df['epoch'].max())
            else:
                axes[i].set_title(f'Metrica {metric} non trovata')

        plt.tight_layout()

        # Salva il grafico nella cartella logs
        destination_path = os.path.join(latest_folder, "logs")
        output_plot_name = os.path.join(destination_path, f"{folder_name}_plot.png")
        plt.savefig(output_plot_name, bbox_inches='tight', dpi=150)
        print(f"Grafico salvato con successo in: {output_plot_name}")

        # Mostra il grafico
        plt.show()

-> Cartella più recente individuata: 20260517_1435
Errore: Nessun file CSV di log trovato in: /content/drive/MyDrive/Advanced_Machine_Learning/output_pesi/20260517_1435/logs/run_*_epochs.csv


In [7]:
# Questa riga scollega il runtime e rilascia la GPU istantaneamente
#from google.colab import runtime
#runtime.unassign()